# Circle Packing

Discrete circle packings via Collins-Stephenson iteration on triangulated half-edge graphs. Two entry points:

- `pack_euclidean(G, boundary_radii=...)` — euclidean packing with prescribed boundary radii.
- `pack_hyperbolic(G, boundary_x_radii=...)` — hyperbolic packing in the Poincaré disk (finite boundary x-radii; horocycles not yet supported).

Both require triangulated, simply-connected (disk-topology) input.

In [ ]:
import sys
from pathlib import Path
# Point at the worktree's eucare (editable install lives in the main repo).
sys.path.insert(0, str(Path('../..').resolve()))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import PatchCollection

import eucare as ec
from eucare.example_graphs import from_tiles
from eucare.example_tilesets import platonic, curved_platonic
from eucare.conway import kis_graph
from eucare.circle_packing import pack_euclidean, pack_hyperbolic


def draw_packing(P, ax=None, title=None, unit_circle=False):
    """Render a packed graph: one circle per vertex at ``v['pos']`` with radius ``v['radius']``."""
    if ax is None:
        _, ax = plt.subplots(figsize=(5, 5))
    circles = []
    centers = []
    for v in P.vertices:
        pos = v['pos']
        if np.iscomplexobj(pos):
            c = (float(pos.real), float(pos.imag))
        else:
            c = (float(pos[0]), float(pos[1]))
        centers.append(c)
        circles.append(plt.Circle(c, float(v['radius'])))
    coll = PatchCollection(circles, facecolors='none', edgecolors='k', linewidth=0.7)
    ax.add_collection(coll)
    cs = np.array(centers)
    radii = np.array([float(v['radius']) for v in P.vertices])
    margin = radii.max() * 1.1
    ax.set_xlim(cs[:, 0].min() - margin, cs[:, 0].max() + margin)
    ax.set_ylim(cs[:, 1].min() - margin, cs[:, 1].max() + margin)
    ax.set_aspect('equal')
    ax.set_axis_off()
    if unit_circle:
        ax.add_artist(plt.Circle((0, 0), 1.0, facecolor='none', edgecolor='#aaa', linestyle='--'))
    if title:
        ax.set_title(title)
    return ax

## 1. Euclidean packing of a regular triangular patch

The `{3, 6}` Platonic tiling. With uniform boundary radius `1.0`, by symmetry every interior radius is also `1.0`.

In [ ]:
G = from_tiles(platonic(3), rings=3)
P = pack_euclidean(G, boundary_radii=1.0)
draw_packing(P, title='Uniform boundary, regular {3,6} lattice')
plt.show()

## 2. Euclidean packing with non-uniform boundary

Same combinatorics, varied boundary radii. Pass a callable so radius depends on vertex position. Note how interior radii adjust to satisfy angle-sum constraints.

In [ ]:
G = from_tiles(platonic(3), rings=3)


def varied_boundary(v):
    # angle around origin of the input position used as a stand-in for vertex id
    p = v['pos']
    theta = np.arctan2(p[1], p[0])
    return 0.5 + 0.4 * np.cos(2 * theta)


P = pack_euclidean(G, boundary_radii=varied_boundary)
draw_packing(P, title='Varied boundary radii (cos 2θ)')
plt.show()

## 3. Non-regular triangulation via `kis`

Take a hexagonal tiling and apply `kis` (triangulate every face from its centroid). The resulting graph has mixed vertex degrees, so a uniform-boundary packing produces visibly varied interior radii.

In [ ]:
G = from_tiles(platonic(6), rings=2)
G = kis_graph()(G, delete_on_border=False)
P = pack_euclidean(G, boundary_radii=1.0)
draw_packing(P, title='kis(hex), uniform boundary')
plt.show()

## 4. Hyperbolic packing in the Poincaré disk

The same combinatorics, packed hyperbolically with boundary x-radii at 0.5. Positions live in the unit disk; you can see how the packing curves and the boundary stays away from the unit circle (because we're not yet packing horocycles).

In [ ]:
G = from_tiles(platonic(3), rings=3)
P = pack_hyperbolic(G, boundary_x_radii=0.5)
draw_packing(P, title='Hyperbolic, boundary x-radii = 0.5', unit_circle=True)
plt.show()

### Boundary x-radii approaching 1 (approximating a maximal packing)

As `boundary_x_radii → 1` the boundary moves toward the unit circle. True `x = 1` (horocycles) isn't supported yet (Möbius translation by a unit-circle point degenerates) but `0.999` gets visibly close.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
G = from_tiles(platonic(3), rings=3)
for ax, x in zip(axes, (0.5, 0.9, 0.999)):
    P = pack_hyperbolic(G, boundary_x_radii=x)
    draw_packing(P, ax=ax, title=f'boundary x = {x}', unit_circle=True)
plt.show()

## 5. Composition: hyperbolic → euclidean reinterpretation

`convert_to_euclidean()` flattens the geometry backend. The positions stay in their stored numeric form (the Poincaré disk is already a subset of ℝ²), and downstream operations treat the graph as a flat euclidean graph.

In [ ]:
G = from_tiles(platonic(3), rings=3)
P_hyp = pack_hyperbolic(G, boundary_x_radii=0.5)
P_eu = P_hyp.copy()
P_eu.convert_to_euclidean()

fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 5))
draw_packing(P_hyp, ax=a1, title='Hyperbolic backend', unit_circle=True)
draw_packing(P_eu, ax=a2, title='After convert_to_euclidean')
plt.show()

## 6. Golden-fixture roundtrip

Load a CirclePack `.p` fixture, repack with `pack_euclidean`, and overlay the two. Should be visually indistinguishable up to gauge (rotation/reflection).

In [ ]:
import sys
from pathlib import Path

FIXTURE = Path('../../tests/fixtures/circlepack').resolve()
sys.path.insert(0, str(FIXTURE))
from _parser import parse_p_file, build_heg_from_flowers  # type: ignore

data = parse_p_file(str(FIXTURE / 'egg_a.p'))
G, idx2v = build_heg_from_flowers(data)
boundary = {idx2v[i]: float(data.radii[i])
            for i, nbrs in data.flowers.items()
            if nbrs[0] != nbrs[-1]}
P = pack_euclidean(G, boundary_radii=boundary, copy_graph=False)

# Visualize ours vs. CirclePack's stored centers (note: alpha/beta gauges differ).
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 5))
draw_packing(P, ax=a1, title='Ours (pack_euclidean)')

# Plot CirclePack's stored layout for comparison.
circles = [plt.Circle(tuple(data.centers[i]), float(data.radii[i]))
           for i in range(data.nodecount)]
a2.add_collection(PatchCollection(circles, facecolors='none', edgecolors='k', linewidth=0.7))
cs = data.centers
margin = data.radii.max() * 1.1
a2.set_xlim(cs[:, 0].min() - margin, cs[:, 0].max() + margin)
a2.set_ylim(cs[:, 1].min() - margin, cs[:, 1].max() + margin)
a2.set_aspect('equal'); a2.set_axis_off()
a2.set_title('CirclePack (egg_a.p)')
plt.show()